# GLM Importance Split-Half Stability Analysis

This notebook computes split-half stability summaries for the GLM-based relative-importance weights used in the GFR neuron clustering / relative-importance analysis. It is intentionally minimal: load fitted GFR parameters, construct the parameter feature matrix, train the same linear softmax classifier used for feature importance, compare grouped importances across random split halves, and save the resulting tables.

## Analysis Summary

This split-half analysis quantifies the stability of GLM-based group importance estimates across random subsets of neurons. For each random split, we train the same classifier on two independent halves and compare the resulting importance weights. The paired half-sample differences estimate sampling variability; because Var(I_A - I_B) ≈ 2σ², SD(I_A - I_B)/sqrt(2) estimates the standard error of a half-sample estimate, while SD(I_A - I_B)/2 approximates the standard error of the full-dataset estimate.

This is a stability/resampling analysis, not a shuffled-label significance test.

In [1]:
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

/home/chyeung2/miniconda3/envs/pytorch-3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [2]:
N_SPLITS = 200
N_RESTARTS_PER_HALF = 1
N_RESTARTS_FULL = 30
RANDOM_SEED = 0
QUICK_TEST = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Match the clustering/relative-importance figure setup.
INSTANT = False
BIN_SIZE = 20
ACTV_BIN_SIZE = 20
EVR2_THRESHOLD = 0.5

if QUICK_TEST:
    N_SPLITS = 10
    N_RESTARTS_PER_HALF = 1
    N_RESTARTS_FULL = 3

print(f"Using device: {DEVICE}")
print(
    f"QUICK_TEST={QUICK_TEST}, N_SPLITS={N_SPLITS}, "
    f"N_RESTARTS_PER_HALF={N_RESTARTS_PER_HALF}, N_RESTARTS_FULL={N_RESTARTS_FULL}"
)

Using device: cuda
QUICK_TEST=False, N_SPLITS=200, N_RESTARTS_PER_HALF=1, N_RESTARTS_FULL=30


## Load GFR Parameters And Labels

In [3]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def gen_dataset(params, threshold=0.5):
    with open("data/labels.pickle", "rb") as f:
        labels = pickle.load(f)

    chosen_ids = filter(lambda x: params[x]["evr2"] > threshold, params.keys())

    dataset = {}
    for cell_id in chosen_ids:
        y = labels[cell_id]
        p = params[cell_id]["params"]

        a = torch.tensor(p["a"]).reshape(-1)
        b = torch.tensor(p["b"]).reshape(-1)
        pc = torch.tensor(p["g"]["poly_coeff"]).reshape(-1)
        gb = torch.tensor(p["g"]["b"]).reshape(-1)
        mc = torch.tensor(p["g"]["max_current"]).reshape(-1)
        mfr = torch.tensor(p["g"]["max_firing_rate"]).reshape(-1)
        x = torch.cat([a, b, pc, gb, mc, mfr])

        dataset[cell_id] = (x, y, params[cell_id]["evr2"])

    return dataset


def get_line_name(df, cell_id):
    return df[df["specimen__id"] == cell_id]["line_name"].to_numpy()[0]


def get_labels(cell_ids):
    df = pd.read_csv("data/metadata.csv")
    line_names = [
        "Pvalb", "Sst", "Vip", "Htr3a", "Ndnf", "Cux2", "Nr5a1", "Ntsr1",
        "Rorb", "Scnn1a", "Tlx3", "Rbp4",
    ]
    new_cell_ids = []
    labels = []

    for cell_id in cell_ids:
        line_name = get_line_name(df, cell_id)
        if type(line_name) != str or "|" in line_name:
            pass
        elif line_name.split("-")[0] in line_names:
            new_cell_ids.append(cell_id)
            labels.append(line_name.split("-")[0])

    types, counts = np.unique(labels, return_counts=True)
    types = types[counts > 20]
    new_cell_ids, labels = zip(*list(filter(lambda x: x[1] in types, zip(new_cell_ids, labels))))

    return np.array(new_cell_ids), np.array(labels)


def get_dataset(dataset=None, instant=False):
    if dataset is None:
        with open("data/dataset.pickle", "rb") as f:
            dataset = pickle.load(f)

    cell_ids, ys = get_labels(dataset.keys())
    xs = []
    for cell_id in cell_ids:
        x = dataset[cell_id][0].tolist()
        c0 = (x[-5] ** 2 - x[-3]) / x[-2]
        c1 = x[-4] ** 2 / x[-2]
        if not instant:
            xs.append(x[:-5] + [c0, c1, x[-1]])
        else:
            xs.append(x[8:-5] + [c0, c1, x[-1]])

    xs = np.array(xs)
    return xs, ys, cell_ids


with open("model/best_params.pickle", "rb") as f:
    all_params = pickle.load(f)

dataset = gen_dataset(all_params[(BIN_SIZE, ACTV_BIN_SIZE)], threshold=EVR2_THRESHOLD)
xs, ys_cre, cell_ids = get_dataset(dataset, instant=INSTANT)
ys_cre = np.array(ys_cre)

inhibitory_lines = np.array(["Pvalb", "Sst", "Vip", "Htr3a", "Ndnf"])
ys_inhibitory = np.array(["inhibitory" if y in inhibitory_lines else "excitatory" for y in ys_cre])

assert xs.ndim == 2
assert len(xs) == len(ys_cre) == len(ys_inhibitory) == len(cell_ids)
assert not np.isnan(xs).any(), "xs contains NaNs"
assert not pd.isna(ys_cre).any(), "Cre-line labels contain NaNs"
assert not pd.isna(ys_inhibitory).any(), "inhibitory/excitatory labels contain NaNs"

print(f"n_samples = {xs.shape[0]}")
print(f"n_features = {xs.shape[1]}")
print("inhibitory/excitatory class counts:")
print(pd.Series(ys_inhibitory).value_counts().sort_index().to_string())
print("\nCre-line class counts:")
print(pd.Series(ys_cre).value_counts().sort_index().to_string())

n_samples = 776
n_features = 19
inhibitory/excitatory class counts:
excitatory    342
inhibitory    434

Cre-line class counts:
Cux2       43
Htr3a     107
Ndnf       61
Nr5a1      60
Ntsr1      51
Pvalb     112
Rbp4       45
Rorb       88
Scnn1a     55
Sst        84
Vip        70


## Feature Groups

In [4]:
if not INSTANT:
    feature_names = [f"alpha_{i + 1}" for i in range(8)] + [f"beta_{i + 1}" for i in range(8)] + ["c0", "c1", "gamma"]
else:
    feature_names = [f"beta_{i + 1}" for i in range(8)] + ["c0", "c1", "gamma"]

assert len(feature_names) == xs.shape[1], (len(feature_names), xs.shape[1])

feature_groups = {
    "input_history_kernel": [i for i, name in enumerate(feature_names) if name.startswith("alpha_")],
    "firing_rate_history_kernel": [i for i, name in enumerate(feature_names) if name.startswith("beta_")],
    "activation": [i for i, name in enumerate(feature_names) if name in {"c0", "c1", "gamma"}],
}

# The Figure 5B setup used here includes all alpha, beta, and activation features.
assigned = [idx for indices in feature_groups.values() for idx in indices]
for group, indices in feature_groups.items():
    assert all(0 <= idx < len(feature_names) for idx in indices), f"Invalid feature index in {group}"
assert sorted(assigned) == list(range(len(feature_names))), "Every feature must belong to exactly one group."
assert len(assigned) == len(set(assigned)), "Feature groups overlap."
assert len(feature_groups["input_history_kernel"]) > 0, "Expected alpha/input-history features."
assert len(feature_groups["firing_rate_history_kernel"]) > 0, "Expected beta/firing-rate-history features."
assert len(feature_groups["activation"]) > 0, "Expected activation features."

for group, indices in feature_groups.items():
    print(f"{group}: {[feature_names[i] for i in indices]}")

input_history_kernel: ['alpha_1', 'alpha_2', 'alpha_3', 'alpha_4', 'alpha_5', 'alpha_6', 'alpha_7', 'alpha_8']
firing_rate_history_kernel: ['beta_1', 'beta_2', 'beta_3', 'beta_4', 'beta_5', 'beta_6', 'beta_7', 'beta_8']
activation: ['c0', 'c1', 'gamma']


## GLM Training And Importance

In [5]:
class ParameterDataset(torch.utils.data.Dataset):
    def __init__(self, xs, ys, label_map):
        self.n_classes = len(label_map)
        self.xs = [torch.tensor(x) for x in xs]
        self.ys = [torch.tensor(label_map[y]) for y in ys]
        self.ys = [F.one_hot(y, num_classes=self.n_classes) for y in self.ys]

        types, counts = np.unique(ys, return_counts=True)
        ws = (1 / counts) / np.sum(1 / counts)
        w_map = {t: w for t, w in zip(types, ws)}
        self.ws = [w_map[y] for y in ys]

    def __len__(self):
        return len(self.xs)

    def __getitem__(self, idx):
        return self.xs[idx], self.ys[idx], self.ws[idx]


def get_predictions(model, dataloader, device):
    ys = []
    ys_pred = []
    model.eval()
    with torch.no_grad():
        for X, y, _ in dataloader:
            X = X.to(torch.float32).to(device)
            y = y.to(device)
            ys_pred.append(model(X).detach().cpu())
            ys.append(y.argmax(dim=1).detach().cpu())
    model.train()
    return torch.cat(ys_pred), torch.cat(ys)


def summary(model, dataloader, device):
    y_pred, y = get_predictions(model, dataloader, device)
    y_pred_class = y_pred.argmax(dim=1).numpy()
    y_true = y.numpy()
    f = f1_score(y_true, y_pred_class, average="weighted", zero_division=0)
    acc = accuracy_score(y_true, y_pred_class)
    return f, acc


def train_model(
    xs,
    ys,
    pca=True,
    scale=True,
    linear=False,
    device=None,
    random_state=None,
    epochs=500,
    evaluate=True,
):
    """Train the GLM/softmax classifier used for the Figure 5B importance weights.

    The preprocessing choices match the original notebook use for Figure 5B: pca=False,
    scale=True, linear=True. The importance calculation below uses the L2 norm of the
    trained linear layer's class weights for each input feature, normalized to sum to 1.
    """
    if random_state is not None:
        set_seed(random_state)

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(device)

    ys = np.array(ys)
    types = np.unique(ys).tolist()
    label_map = {t: i for i, t in enumerate(types)}
    n_classes = len(label_map)

    indices = np.arange(len(xs))
    idx_tr, idx_te = train_test_split(
        indices, test_size=0.35, stratify=ys, random_state=random_state
    )
    idx_tr, idx_val = train_test_split(
        idx_tr, test_size=0.15, stratify=ys[idx_tr], random_state=random_state
    )

    Xtr, ytr = xs[idx_tr], ys[idx_tr]
    Xval, yval = xs[idx_val], ys[idx_val]
    Xte, yte = xs[idx_te], ys[idx_te]

    if scale:
        scaler = StandardScaler()
        scaler.fit(Xtr)
        Xtr = scaler.transform(Xtr)
        Xval = scaler.transform(Xval)
        Xte = scaler.transform(Xte)
        xs_model = scaler.transform(xs)
    else:
        xs_model = xs

    if pca:
        scaler = PCA(n_components=5, random_state=random_state)
        scaler.fit(Xtr)
        Xtr = scaler.transform(Xtr)
        Xval = scaler.transform(Xval)
        Xte = scaler.transform(Xte)
        xs_model = scaler.transform(xs_model)

    train_dataset = ParameterDataset(Xtr, ytr, label_map)
    val_dataset = ParameterDataset(Xval, yval, label_map)
    test_dataset = ParameterDataset(Xte, yte, label_map)

    generator = torch.Generator()
    if random_state is not None:
        generator.manual_seed(random_state)

    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=64, shuffle=True, generator=generator
    )
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=64, shuffle=False)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=len(yte), shuffle=False)

    if linear:
        model = torch.nn.Sequential(torch.nn.Linear(len(xs_model[0]), n_classes))
    else:
        model = torch.nn.Sequential(
            torch.nn.Linear(len(xs_model[0]), 10),
            torch.nn.ReLU(),
            torch.nn.Linear(10, n_classes),
        )
    model = model.to(device)

    types, counts = np.unique(ys, return_counts=True)
    ws = torch.tensor((1 / counts) / np.sum(1 / counts), dtype=torch.float32, device=device)

    criterion = torch.nn.CrossEntropyLoss(weight=ws)
    optim = torch.optim.Adam(model.parameters(), lr=0.005)

    for _ in range(epochs):
        loss = 0
        for X, y, _ in train_loader:
            X = X.to(torch.float32).to(device)
            y = y.to(torch.float32).to(device)
            y_pred = model(X)
            loss = loss + criterion(y_pred, y)

        optim.zero_grad()
        loss.backward()
        optim.step()

    if evaluate:
        f, acc = summary(model, test_loader, device)
    else:
        f, acc = np.nan, np.nan

    return model, f, acc


def feature_importance_from_model(model):
    weights = model[0].weight.detach().norm(dim=0).cpu().numpy()
    weights = weights / weights.sum()
    assert np.isclose(weights.sum(), 1.0, atol=1e-6), weights.sum()
    return weights


def compute_feature_importance(xs, labels, n_restarts, base_seed, desc=None, evaluate=False):
    labels = np.array(labels)
    importances = []
    f1s = []
    accs = []

    iterator = range(n_restarts)
    if desc is not None:
        iterator = tqdm(iterator, desc=desc, leave=False)

    for restart in iterator:
        model, f, acc = train_model(
            xs,
            labels,
            pca=False,
            scale=True,
            linear=True,
            device=DEVICE,
            random_state=base_seed + restart,
            evaluate=evaluate,
        )
        importances.append(feature_importance_from_model(model))
        f1s.append(f)
        accs.append(acc)

    importances = np.stack(importances)
    mean_importance = importances.mean(axis=0)
    assert np.isclose(mean_importance.sum(), 1.0, atol=1e-6), mean_importance.sum()
    return mean_importance, importances, np.array(f1s), np.array(accs)


def group_importances(feature_importance, feature_groups):
    return {
        group: float(np.mean(feature_importance[indices]))
        for group, indices in feature_groups.items()
    }

## Full-Dataset Importance Weights

In [6]:
targets = {
    "inhibitory_vs_excitatory": ys_inhibitory.copy(),
    "cre_line": ys_cre.copy(),
}

group_names = list(feature_groups.keys())
full_feature_importances = {}
full_restart_importances = {}
full_group_importances = {}
full_metrics = {}

for target_i, (target_name, labels) in enumerate(targets.items()):
    labels = np.array(labels)
    mean_importance, restart_importances, f1s, accs = compute_feature_importance(
        xs,
        labels,
        n_restarts=N_RESTARTS_FULL,
        base_seed=RANDOM_SEED + 10_000 * target_i,
        desc=f"full data {target_name}",
        evaluate=True,
    )
    full_feature_importances[target_name] = mean_importance
    full_restart_importances[target_name] = restart_importances
    full_group_importances[target_name] = group_importances(mean_importance, feature_groups)
    full_metrics[target_name] = {"f1_mean": np.nanmean(f1s), "acc_mean": np.nanmean(accs)}

    print(f"{target_name}: full-data GLM importance")
    print(f"  mean F1={np.nanmean(f1s):.4f}, mean accuracy={np.nanmean(accs):.4f}")
    for group, value in full_group_importances[target_name].items():
        print(f"  {group}: {value:.6f}")

inhibitory_vs_excitatory: full-data GLM importance
  mean F1=0.7814, mean accuracy=0.7810
  input_history_kernel: 0.047776
  firing_rate_history_kernel: 0.048975
  activation: 0.075331


cre_line: full-data GLM importance
  mean F1=0.3324, mean accuracy=0.3544
  input_history_kernel: 0.048617
  firing_rate_history_kernel: 0.051787
  activation: 0.065591


## Split-Half Stability Analysis

In [7]:
def can_stratify(labels):
    _, counts = np.unique(labels, return_counts=True)
    return len(counts) > 1 and np.all(counts >= 2)


def split_half_indices(labels, rng):
    labels = np.array(labels)
    indices = np.arange(len(labels))
    if can_stratify(labels):
        split_seed = int(rng.integers(0, np.iinfo(np.int32).max))
        idx_a, idx_b = train_test_split(
            indices,
            test_size=0.5,
            stratify=labels,
            random_state=split_seed,
        )
        return np.array(idx_a), np.array(idx_b), True

    shuffled = indices.copy()
    rng.shuffle(shuffled)
    midpoint = len(shuffled) // 2
    return shuffled[:midpoint], shuffled[midpoint:], False


def as_group_vector(group_values):
    return np.array([group_values[group] for group in group_names], dtype=float)


def pearson_corr(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if len(a) < 2 or np.isclose(np.std(a), 0) or np.isclose(np.std(b), 0):
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def spearman_corr(a, b):
    rank_a = pd.Series(a).rank(method="average").to_numpy(dtype=float)
    rank_b = pd.Series(b).rank(method="average").to_numpy(dtype=float)
    return pearson_corr(rank_a, rank_b)


rng = np.random.default_rng(RANDOM_SEED)
raw_rows = []
stability_rows = []
pairwise_rows = []

pairwise_specs = [
    ("activation_minus_input_history_kernel", "activation", "input_history_kernel"),
    ("activation_minus_firing_rate_history_kernel", "activation", "firing_rate_history_kernel"),
    ("input_history_minus_firing_rate_history_kernel", "input_history_kernel", "firing_rate_history_kernel"),
]

for target_i, (target_name, labels) in enumerate(targets.items()):
    labels = np.array(labels)
    print(f"{target_name}: split-half class counts")
    print(pd.Series(labels).value_counts().sort_index().to_string())
    print(f"  stratified splitting possible: {can_stratify(labels)}")

    stratified_flags = []
    for split_i in tqdm(range(N_SPLITS), desc=f"split halves {target_name}"):
        idx_a, idx_b, stratified = split_half_indices(labels, rng)
        stratified_flags.append(stratified)

        imp_a, _, _, _ = compute_feature_importance(
            xs[idx_a],
            labels[idx_a],
            n_restarts=N_RESTARTS_PER_HALF,
            base_seed=RANDOM_SEED + 1_000_000 + 100_000 * target_i + 100 * split_i,
            evaluate=False,
        )
        imp_b, _, _, _ = compute_feature_importance(
            xs[idx_b],
            labels[idx_b],
            n_restarts=N_RESTARTS_PER_HALF,
            base_seed=RANDOM_SEED + 2_000_000 + 100_000 * target_i + 100 * split_i,
            evaluate=False,
        )

        group_a = group_importances(imp_a, feature_groups)
        group_b = group_importances(imp_b, feature_groups)
        vector_a = as_group_vector(group_a)
        vector_b = as_group_vector(group_b)

        stability_rows.append(
            {
                "target": target_name,
                "split": split_i,
                "pearson_r": pearson_corr(vector_a, vector_b),
                "spearman_r": spearman_corr(vector_a, vector_b),
                "stratified": stratified,
                "n_A": len(idx_a),
                "n_B": len(idx_b),
            }
        )

        for group in group_names:
            diff = group_a[group] - group_b[group]
            raw_rows.append(
                {
                    "target": target_name,
                    "split": split_i,
                    "group": group,
                    "I_A": group_a[group],
                    "I_B": group_b[group],
                    "diff": diff,
                    "abs_diff": abs(diff),
                    "stratified": stratified,
                    "n_A": len(idx_a),
                    "n_B": len(idx_b),
                }
            )

        for comparison, group_1, group_2 in pairwise_specs:
            # One paired split-level contrast, averaging the contrast from the two halves.
            split_difference = 0.5 * (
                (group_a[group_1] - group_a[group_2])
                + (group_b[group_1] - group_b[group_2])
            )
            pairwise_rows.append(
                {
                    "target": target_name,
                    "split": split_i,
                    "comparison": comparison,
                    "group_1": group_1,
                    "group_2": group_2,
                    "difference": split_difference,
                    "positive": split_difference > 0,
                }
            )

    print(f"  stratified splits used: {sum(stratified_flags)} / {len(stratified_flags)}")

raw_split_half = pd.DataFrame(raw_rows)
stability_by_split = pd.DataFrame(stability_rows)
pairwise_by_split = pd.DataFrame(pairwise_rows)

raw_split_half.head()

inhibitory_vs_excitatory: split-half class counts
excitatory    342
inhibitory    434
  stratified splitting possible: True


split halves inhibitory_vs_excitatory: 100%|██████████| 200/200 [21:23<00:00,  6.42s/it]


  stratified splits used: 200 / 200
cre_line: split-half class counts
Cux2       43
Htr3a     107
Ndnf       61
Nr5a1      60
Ntsr1      51
Pvalb     112
Rbp4       45
Rorb       88
Scnn1a     55
Sst        84
Vip        70
  stratified splitting possible: True


split halves cre_line: 100%|██████████| 200/200 [21:23<00:00,  6.42s/it]

  stratified splits used: 200 / 200


,target,split,group,I_A,I_B,diff,abs_diff,stratified,n_A,n_B
0,inhibitory_vs_excitatory,0,input_history_kernel,0.041422,0.046475,-0.005053,0.005053,True,388,388
1,inhibitory_vs_excitatory,0,firing_rate_history_kernel,0.057715,0.056589,0.001126,0.001126,True,388,388
2,inhibitory_vs_excitatory,0,activation,0.068969,0.058497,0.010472,0.010472,True,388,388
3,inhibitory_vs_excitatory,1,input_history_kernel,0.066234,0.035754,0.030479,0.030479,True,388,388
4,inhibitory_vs_excitatory,1,firing_rate_history_kernel,0.035844,0.060763,-0.024919,0.024919,True,388,388


## Summary Tables

In [8]:
stability_summary = (
    stability_by_split.groupby("target", sort=False)
    .agg(
        pearson_r_mean=("pearson_r", "mean"),
        pearson_r_std=("pearson_r", "std"),
        spearman_r_mean=("spearman_r", "mean"),
        spearman_r_std=("spearman_r", "std"),
        n_splits=("split", "nunique"),
        stratified_splits=("stratified", "sum"),
    )
    .reset_index()
)

summary_rows = []
for target_name in targets:
    stability_row = stability_summary.set_index("target").loc[target_name].to_dict()
    for group in group_names:
        group_rows = raw_split_half[
            (raw_split_half["target"] == target_name) & (raw_split_half["group"] == group)
        ]
        all_half_estimates = np.concatenate(
            [group_rows["I_A"].to_numpy(dtype=float), group_rows["I_B"].to_numpy(dtype=float)]
        )
        sd_diff = float(group_rows["diff"].std(ddof=1)) if len(group_rows) > 1 else np.nan
        se_half = sd_diff / np.sqrt(2) if np.isfinite(sd_diff) else np.nan
        se_full_approx = sd_diff / 2 if np.isfinite(sd_diff) else np.nan
        observed = full_group_importances[target_name][group]
        summary_rows.append(
            {
                "target": target_name,
                "group": group,
                "observed_importance": observed,
                "mean_importance_half_A": float(group_rows["I_A"].mean()),
                "mean_importance_half_B": float(group_rows["I_B"].mean()),
                "mean_importance_all_halves": float(np.mean(all_half_estimates)),
                "sd_all_halves": float(np.std(all_half_estimates, ddof=1)),
                "sd_diff": sd_diff,
                "se_half": se_half,
                "se_full_approx": se_full_approx,
                "ci95_low": observed - 1.96 * se_full_approx,
                "ci95_high": observed + 1.96 * se_full_approx,
                "pearson_r_mean": stability_row["pearson_r_mean"],
                "pearson_r_std": stability_row["pearson_r_std"],
                "spearman_r_mean": stability_row["spearman_r_mean"],
                "spearman_r_std": stability_row["spearman_r_std"],
                "n_splits": N_SPLITS,
                "stratified_splits": int(stability_row["stratified_splits"]),
                "n_restarts_full": N_RESTARTS_FULL,
                "n_restarts_per_half": N_RESTARTS_PER_HALF,
            }
        )

summary = pd.DataFrame(summary_rows)

pairwise_summary = (
    pairwise_by_split.groupby(["target", "comparison", "group_1", "group_2"], sort=False)
    .agg(
        mean_difference=("difference", "mean"),
        std_difference=("difference", "std"),
        fraction_positive=("positive", "mean"),
        n_splits=("split", "nunique"),
    )
    .reset_index()
)

print("Split-half stability correlations:")
display(stability_summary)
print("Group-importance summary:")
display(summary)
print("Pairwise group comparisons; these are stability/resampling summaries, not formal p-values:")
display(pairwise_summary)

Split-half stability correlations:


,target,pearson_r_mean,pearson_r_std,spearman_r_mean,spearman_r_std,n_splits,stratified_splits
0,inhibitory_vs_excitatory,0.520422,0.519943,0.4400,0.579291,200,200
1,cre_line,0.940410,0.080376,0.8325,0.236587,200,200


Group-importance summary:


,target,group,observed_importance,mean_importance_half_A,mean_importance_half_B,mean_importance_all_halves,sd_all_halves,sd_diff,se_half,se_full_approx,ci95_low,ci95_high,pearson_r_mean,pearson_r_std,spearman_r_mean,spearman_r_std,n_splits,stratified_splits,n_restarts_full,n_restarts_per_half
0,inhibitory_vs_excitatory,input_history_kernel,0.047776,0.045159,0.044368,0.044763,0.007954,0.011970,0.008464,0.005985,0.036046,0.059507,0.520422,0.519943,0.4400,0.579291,200,200,30,1
1,inhibitory_vs_excitatory,firing_rate_history_kernel,0.048975,0.051496,0.053649,0.052572,0.008699,0.013390,0.009468,0.006695,0.035852,0.062097,0.520422,0.519943,0.4400,0.579291,200,200,30,1
2,inhibitory_vs_excitatory,activation,0.075331,0.075587,0.071955,0.073771,0.015088,0.024574,0.017376,0.012287,0.051248,0.099413,0.520422,0.519943,0.4400,0.579291,200,200,30,1
3,cre_line,input_history_kernel,0.048617,0.046856,0.046791,0.046824,0.002982,0.004327,0.003060,0.002164,0.044376,0.052857,0.940410,0.080376,0.8325,0.236587,200,200,30,1
4,cre_line,firing_rate_history_kernel,0.051787,0.051338,0.051279,0.051309,0.002867,0.004356,0.003080,0.002178,0.047518,0.056055,0.940410,0.080376,0.8325,0.236587,200,200,30,1
5,cre_line,activation,0.065591,0.071483,0.071812,0.071648,0.005633,0.008731,0.006174,0.004365,0.057035,0.074148,0.940410,0.080376,0.8325,0.236587,200,200,30,1


Pairwise group comparisons; these are stability/resampling summaries, not formal p-values:


,target,comparison,group_1,group_2,mean_difference,std_difference,fraction_positive,n_splits
0,inhibitory_vs_excitatory,activation_minus_input_history_kernel,activation,input_history_kernel,0.029008,0.011076,0.995,200
1,inhibitory_vs_excitatory,activation_minus_firing_rate_history_kernel,activation,firing_rate_history_kernel,0.021199,0.011747,0.970,200
2,inhibitory_vs_excitatory,input_history_minus_firing_rate_history_kernel,input_history_kernel,firing_rate_history_kernel,-0.007809,0.010209,0.260,200
3,cre_line,activation_minus_input_history_kernel,activation,input_history_kernel,0.024824,0.004869,1.000,200
4,cre_line,activation_minus_firing_rate_history_kernel,activation,firing_rate_history_kernel,0.020339,0.004361,1.000,200
5,cre_line,input_history_minus_firing_rate_history_kernel,input_history_kernel,firing_rate_history_kernel,-0.004485,0.003697,0.095,200


## Save Tables

In [9]:
summary_csv_path = Path("cluster_importance_split_half_summary.csv")
summary_tex_path = Path("cluster_importance_split_half_summary.tex")
pairwise_csv_path = Path("cluster_importance_split_half_pairwise.csv")
pairwise_tex_path = Path("cluster_importance_split_half_pairwise.tex")
raw_csv_path = Path("cluster_importance_split_half_raw.csv")

summary.to_csv(summary_csv_path, index=False)
summary.to_latex(summary_tex_path, index=False, float_format="%.4g")
pairwise_summary.to_csv(pairwise_csv_path, index=False)
pairwise_summary.to_latex(pairwise_tex_path, index=False, float_format="%.4g")
raw_split_half.to_csv(raw_csv_path, index=False)

print(f"Saved {summary_csv_path}")
print(f"Saved {summary_tex_path}")
print(f"Saved {pairwise_csv_path}")
print(f"Saved {pairwise_tex_path}")
print(f"Saved {raw_csv_path}")

Saved cluster_importance_split_half_summary.csv
Saved cluster_importance_split_half_summary.tex
Saved cluster_importance_split_half_pairwise.csv
Saved cluster_importance_split_half_pairwise.tex
Saved cluster_importance_split_half_raw.csv


## Copy-Ready Manuscript Text

In [10]:
method_paragraph = (
    "To quantify the stability of the GLM-based relative-importance trends, we performed "
    "repeated split-half analyses. For each classification target, neurons were randomly "
    "divided into two approximately equal class-stratified halves when class counts allowed. "
    "The same linear softmax classifier used for the main relative-importance analysis was "
    "trained independently on each half, and feature importance was computed as the L2 norm "
    "of each feature's class weights normalized to sum to one. Parameters were grouped into "
    "input-history kernel, firing-rate-history kernel, and activation parameters, with group "
    "importance defined as the mean normalized feature importance within each group. Across "
    f"{N_SPLITS} repeated splits, paired half-sample differences were used to estimate "
    "sampling variability: SD(I_A-I_B)/sqrt(2) estimates the standard error of a half-sample "
    "estimate, and SD(I_A-I_B)/2 approximates the standard error of the full-dataset estimate. "
    "The analysis provides stability and resampling evidence for the observed trends."
)

compact_columns = [
    "target",
    "group",
    "observed_importance",
    "se_full_approx",
    "ci95_low",
    "ci95_high",
    "pearson_r_mean",
    "spearman_r_mean",
]
compact_table = summary[compact_columns]

print(method_paragraph)
print("\nCompact LaTeX table:")
print(compact_table.to_latex(index=False, float_format="%.4g"))

To quantify the stability of the GLM-based relative-importance trends, we performed repeated split-half analyses. For each classification target, neurons were randomly divided into two approximately equal class-stratified halves when class counts allowed. The same linear softmax classifier used for the main relative-importance analysis was trained independently on each half, and feature importance was computed as the L2 norm of each feature's class weights normalized to sum to one. Parameters were grouped into input-history kernel, firing-rate-history kernel, and activation parameters, with group importance defined as the mean normalized feature importance within each group. Across 200 repeated splits, paired half-sample differences were used to estimate sampling variability: SD(I_A-I_B)/sqrt(2) estimates the standard error of a half-sample estimate, and SD(I_A-I_B)/2 approximates the standard error of the full-dataset estimate. The analysis provides stability and resampling evidence f